# Fine-tune DistilBERT on all 8 Kaggle ASAP Datasets

This notebook scales the existing transformer training pipeline to utilize all eight essay sets from the Kaggle ASAP dataset. It incorporates independent rescaling for each set to ensure a unified 0-100 target variable, aiming to reduce the MAE to the 3-4 range.

In [1]:
!wget https://raw.githubusercontent.com/Turanga1/Automated-Essay-Scoring/master/training_set_rel3.tsv

--2026-07-27 12:56:38--  https://raw.githubusercontent.com/Turanga1/Automated-Essay-Scoring/master/training_set_rel3.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16324188 (16M) [application/octet-stream]
Saving to: ‘training_set_rel3.tsv’

training_set_rel3.t 100%[===================>]  15.57M  --.-KB/s    in 0.09s   

2026-07-27 12:56:39 (168 MB/s) - ‘training_set_rel3.tsv’ saved [16324188/16324188]



In [3]:
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [2]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [4]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Ensure GPU is utilized if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
# 1. Load and prepare all 8 Kaggle ASAP datasets
# The standard Kaggle ASAP file 'training_set_rel3.tsv' contains all 8 sets.
CSV_PATH = "training_set_rel3.tsv" # Ensure this file is downloaded from Kaggle

try:
    df = pd.read_csv(CSV_PATH, sep='\t', encoding='latin-1')
except FileNotFoundError:
    print(f"Please download {CSV_PATH} from Kaggle and place it in the working directory.")
    df = pd.DataFrame(columns=['essay_id', 'essay_set', 'essay', 'domain1_score'])

if not df.empty:
    df = df[['essay_id', 'essay_set', 'essay', 'domain1_score']].dropna()

    rescaled_dfs = []
    
    # Iterate through all 8 datasets to normalize their unique rubrics to 0-100
    for i in range(1, 9):
        set_df = df[df['essay_set'] == i].copy()
        if set_df.empty: continue
            
        min_val = set_df['domain1_score'].min()
        max_val = set_df['domain1_score'].max()
        
        # Rescale current set to 0-100
        set_df['score'] = ((set_df['domain1_score'] - min_val) / (max_val - min_val)) * 100
        rescaled_dfs.append(set_df)

    full_df = pd.concat(rescaled_dfs, ignore_index=True)
    
    # Rescale to 0-1 for model training stability
    full_df['label'] = full_df['score'] / 100.0
    print(f"Total essays loaded and normalized across all ASAP datasets: {len(full_df)}")

Total essays loaded and normalized across all ASAP datasets: 12976


In [6]:
# 2. Train / validation split
train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} | Validation: {len(val_df)}")

Train: 10380 | Validation: 2596


In [7]:
# 3. Tokenize using DistilBERT
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[['essay', 'label']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['essay', 'label']], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["essay"], truncation=True, padding="max_length", max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10380 [00:00<?, ? examples/s]

Map:   0%|          | 0/2596 [00:00<?, ? examples/s]

In [8]:
# 4. Load the model with a regression head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# 5. Setup Trainer and Hyperparameters
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    # Rescale back to 0-100 scale to check against the 3-4 target MAE
    mae = mean_absolute_error(labels * 100, preds * 100)
    return {"mae_0_100": mae}

training_args = TrainingArguments(
    output_dir="./results_all_asap",
    num_train_epochs=3,
    per_device_train_batch_size=16, # Increased from 8 to handle larger corpus
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae_0_100",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [10]:
# 6. Train and Evaluate against the target MAE (3-4)
trainer.train()

metrics = trainer.evaluate()
print("\n--- Evaluation Results ---")
print(f"Transformer MAE (All 8 ASAP Datasets): {metrics['eval_mae_0_100']:.2f}")
print("Target MAE Range: 3.00 - 4.00")

# 7. Save Model
SAVE_DIR = "transformer_model_full_asap"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\nModel successfully saved to {SAVE_DIR}. Swap this into backend/transformer_model/.")

Epoch,Training Loss,Validation Loss,Mae 0 100
1,0.023798,0.019914,10.918883
2,0.018020,0.017799,10.126912
3,0.014604,0.017507,9.966913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Mae 0 100
0.014604,0.017507,3,9.966913



--- Evaluation Results ---
Transformer MAE (All 8 ASAP Datasets): 9.97
Target MAE Range: 3.00 - 4.00


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model successfully saved to transformer_model_full_asap. Swap this into backend/transformer_model/.


In [11]:
# 1. Zip the trained model directory
!zip -r transformer_model_full_asap.zip transformer_model_full_asap

# 2. Trigger browser download (if supported by your current extension view)
from google.colab import files
files.download("transformer_model_full_asap.zip")

  adding: transformer_model_full_asap/ (stored 0%)
  adding: transformer_model_full_asap/config.json (deflated 49%)
  adding: transformer_model_full_asap/tokenizer_config.json (deflated 43%)
  adding: transformer_model_full_asap/model.safetensors (deflated 8%)
  adding: transformer_model_full_asap/tokenizer.json (deflated 71%)
  adding: transformer_model_full_asap/training_args.bin (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!curl --progress-bar --upload-file ./transformer_model_full_asap.zip https://transfer.sh/transformer_model_full_asap.zip

curl: (7) Failed to connect to transfer.sh port 443 after 173 ms: Connection refused


In [13]:
from google.colab import drive

# 1. Mount your Google Drive (it will prompt you to click a link to authorize)
drive.mount('/content/drive')

# 2. Copy the zip file directly to the root of your Google Drive
!cp ./transformer_model_full_asap.zip /content/drive/MyDrive/

print("Copy complete! Check your Google Drive.")

Mounted at /content/drive
Copy complete! Check your Google Drive.
